# 🧪 DevSRE-Companion: Fine-Tuning com LoRA / QLoRA no Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Este notebook especializa um modelo leve (**Gemma 2 2B / Llama 3.2 3B**) na tarefa de **Root Cause Analysis (RCA)** para incidentes críticos de engenharia de software e SRE.

### 🎯 Por que LoRA / QLoRA?
- **Economia Brutal de VRAM:** Quantização de 4 bits (QLoRA) reduz o consumo de memória de ~16GB para menos de 4GB, rodando perfeitamente em GPUs gratuitas ou Colab Pro (T4, L4, A100).
- **Preservação de Conhecimento:** Os pesos originais do modelo são congelados ($W_0$), treinando apenas matrizes de baixo posto ($A$ e $B$, onde $\Delta W = B \times A$).
- **Formatação Rigorosa:** O modelo aprende a estruturar diagnósticos técnicos, apontar arquivos afetados e propor planos de ação preventivos sem prolixidade.

In [ ]:
# 1. Verificar GPU alocada pelo Google Colab
!nvidia-smi

In [ ]:
# 2. Instalar dependências essenciais de Deep Learning e Fine-Tuning
!pip install -q -U torch transformers datasets peft trl bitsandbytes accelerate matplotlib

## 📂 3. Carregar e Inspecionar o Dataset de Incidentes (`rca_dataset.jsonl`)

In [ ]:
import json
from datasets import load_dataset

# Se estiver executando no Colab, faça upload do rca_dataset.jsonl na barra lateral
DATASET_PATH = 'rca_dataset.jsonl'

dataset = load_dataset('json', data_files=DATASET_PATH)
print(f'Total de amostras de treino: {len(dataset["train"])}')
print('\n--- Exemplo de Amostra ---')
print(json.dumps(dataset['train'][0], indent=2, ensure_ascii=False))

## 🧠 4. Configuração do Modelo Base e Quantização QLoRA (4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'google/gemma-2-2b-it'  # Ou 'meta-llama/Llama-3.2-3B-Instruct'

# Configuração de Quantização NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

# Configuração dos Adaptadores LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
print('\n--> Parâmetros Treináveis:')
model.print_trainable_parameters()

## 🚀 5. Treinamento com TRL SFTTrainer

In [ ]:
from transformers import TrainingArguments
try:
    from trl import SFTConfig
    config_cls = SFTConfig
except ImportError:
    config_cls = TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = './devsre_rca_adapter'

# Aplica chat template do modelo aos dados
def format_chat_prompt(batch):
    return {'text': [tokenizer.apply_chat_template(m, tokenize=False) for m in batch['messages']]}

dataset = dataset.map(format_chat_prompt, batched=True)

training_args = config_cls(
    output_dir=OUTPUT_DIR,
    dataset_text_field='text' if hasattr(config_cls, 'dataset_text_field') else None,
    max_seq_length=1024 if hasattr(config_cls, 'max_seq_length') else None,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    logging_steps=1,
    max_steps=40,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    fp16=not torch.cuda.is_bf16_supported(),
    optim='paged_adamw_8bit',
    save_strategy='steps',
    save_steps=20,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    peft_config=lora_config,
    args=training_args,
)

print('--> Iniciando treinamento no Colab...')
trainer.train()
print('--> Treinamento concluído com sucesso!')

## 📈 6. Visualização da Curva de Aprendizado (Loss Curve)

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [entry['step'] for entry in log_history if 'loss' in entry]
losses = [entry['loss'] for entry in log_history if 'loss' in entry]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker='o', color='#2563eb', label='Training Loss')
plt.title('Curva de Aprendizado - DevSRE RCA LoRA')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## 🧪 7. Avaliação Comparativa: Modelo com Adaptador LoRA em Novo Incidente Inédito

In [ ]:
prompt_teste = """<bos><start_of_turn>user
Incidente: Timeout em lote de processamento de mensagens SQS.
Erro: botocore.exceptions.ClientError: An error occurred (OverLimit) when calling ReceiveMessage.
Runbook: runbook_queue_sqs.md exige backoff exponencial e limite de visibilidade de 60s.
Gere o relatório de RCA estrito.<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(prompt_teste, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.2)

resposta_gerada = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('=== Resposta Gerada pelo Modelo com LoRA ===')
print(resposta_gerada)

## 💾 8. Salvar e Exportar os Adaptadores LoRA

In [ ]:
# Salvar adaptadores e criar arquivo ZIP para download
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

!zip -r devsre_rca_adapter.zip ./devsre_rca_adapter
print('--> Arquivo devsre_rca_adapter.zip pronto para download no Google Colab!')